Step 1: Define File Path and Load Data
This step defines the file path for the 2024 property assessment dataset and attempts to load the file. If found, it loads the dataset, removes any leading/trailing whitespace from column names, and handles missing values to ensure consistency.

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [2]:
# Load the dataset
file_path = r'C:\Users\sul19\Desktop\701 Project\Property Assessment Datasets\‏‏Property Assessment 2024 V1.xlsx'

# Attempt to load with 'openpyxl' engine
df = pd.read_excel(file_path, engine='openpyxl', na_values=['', ' '])

# Strip any leading/trailing whitespace from the column names (just in case)
df.columns = df.columns.str.strip()

Step 2: Standardize Values in OVERALL_COND Column
This step removes non-breaking spaces and other invisible characters from the OVERALL_COND column, then replaces any blank or missing values with "none" to ensure consistency. It also standardizes specific values for improved data quality in the next steps.

In [3]:
# Replace non-breaking spaces and other invisible characters in the overall condition column
df['OVERALL_COND'] = df['OVERALL_COND'].astype(str).str.replace('\u00A0', '').str.strip()

# Replace any blank or missing values with 'none'
df['OVERALL_COND'] = df['OVERALL_COND'].replace(r'^\s*$', 'none', regex=True)

# Replace specific values in OVERALL_COND
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'AVG - Default - Average': 'A - Average',
    'EX - Excellent': 'E - Excellent'
}, regex=False)

Step 3: Group Data by ZIP_CODE and Summarize OVERALL_COND Counts
In this step, the data is grouped by ZIP code, counting occurrences of each condition. This summary provides an overview of housing conditions across different areas based on condition counts.

In [4]:
# Group data by 'ZIP_CODE' and count the occurrences of each condition in the column
overall_cond_summary = df.groupby('ZIP_CODE')['OVERALL_COND'].value_counts().unstack().fillna(0)

# Display the result of the analysis
print("Housing condition summary by ZIP code:")
#print(condition_summary)
print(overall_cond_summary)

Housing condition summary by ZIP code:
OVERALL_COND  A - Average  E - Excellent  F - Fair  G - Good  P - Poor  \
ZIP_CODE                                                                 
2026.0                5.0            0.0       0.0       1.0       0.0   
2108.0             1545.0          148.0       8.0     396.0       0.0   
2109.0             1445.0           16.0       4.0     307.0       0.0   
2110.0             2006.0          305.0       4.0      78.0       1.0   
2111.0             1886.0          314.0      22.0     489.0       0.0   
2113.0             1673.0            6.0      27.0     501.0       2.0   
2114.0             4363.0           62.0       7.0     774.0       1.0   
2115.0             4158.0          471.0       4.0     813.0       0.0   
2116.0             6857.0          374.0      32.0    1874.0       7.0   
2118.0             5807.0          174.0      26.0    2756.0       6.0   
2119.0             4292.0            5.0      87.0    1286.0      19.0   

Step 4: Map Condition Labels to Numeric Values for Analysis
To facilitate quantitative analysis, this step maps condition labels to numeric values. We then calculate the mean condition score for each ZIP code, offering insights into the average housing condition in each area.

Step 5: Convert Mean Condition Scores to Descriptive Labels
This step converts mean condition scores back to descriptive labels for better readability. Each ZIP code receives a condition label that represents the general state of housing based on its average condition score.

In [5]:
# Function to assign numerical values to conditions 
def condition_to_numeric(cond):
    mapping = {
        'E - Excellent': 5,
        'VG - Very Good': 4,
        'G - Good': 3.5,
        'A - Average': 3,
        'F - Fair': 2,
        'P - Poor': 1.5,
        'VP - Very Poor': 1,
        'US - Unsound': 0,
        
        
        'none': np.nan  # Treat 'none' as NaN for numerical purposes
    }
    return mapping.get(cond, np.nan)

# Apply the mapping to calculate average conditions
df['OVERALL_COND_NUM'] = df['OVERALL_COND'].apply(condition_to_numeric)

# Group by ZIP_CODE and calculate the mean, count, and standard deviation
condition_analysis = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Display the summary
print("Housing condition analysis by ZIP code:")
print(condition_analysis)

def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    
    else:
        return 'Unsound'

# Apply the function to map the means back to descriptive condition labels
condition_analysis['overall_cond_label'] = condition_analysis['overall_cond_mean'].apply(mean_to_condition)

# Print the result for each ZIP code
print("Overall Condition Analysis by ZIP Code:")
print(condition_analysis[['ZIP_CODE', 'overall_cond_mean', 'overall_cond_label']])

Housing condition analysis by ZIP code:
    ZIP_CODE  overall_cond_mean
0     2026.0           3.083333
1     2108.0           3.251739
2     2109.0           3.124656
3     2110.0           3.280813
4     2111.0           3.329925
5     2113.0           3.112882
6     2114.0           3.104527
7     2115.0           3.255761
8     2116.0           3.211530
9     2118.0           3.210819
10    2119.0           3.098952
11    2120.0           3.115517
12    2121.0           3.090680
13    2122.0           3.104991
14    2124.0           3.098206
15    2125.0           3.115815
16    2126.0           3.050610
17    2127.0           3.151033
18    2128.0           3.134756
19    2129.0           3.195933
20    2130.0           3.149320
21    2131.0           3.101358
22    2132.0           3.095967
23    2133.0           4.000000
24    2134.0           3.070128
25    2135.0           3.081545
26    2136.0           3.062111
27    2137.0           3.500000
28    2199.0           4.180556


Step 6: Standardize YR_BUILT and YR_REMODEL Columns
This step ensures YR_BUILT and YR_REMODEL columns are numeric and removes any values beyond 2024 to avoid future-dated entries. The mean construction and remodel years by ZIP code are then calculated for further analysis.

Step 7: Classify Buildings by Age
In this step, buildings are classified based on their construction or remodel year as 'Old,' 'Average,' or 'New.' This classification adds insights into the age distribution within each ZIP code, enhancing the overall housing analysis.

In [6]:
# Ensure YR_BUILT and YR_REMODEL are numeric and replace years > 2024 with NaN
df['YR_BUILT'] = pd.to_numeric(df['YR_BUILT'], errors='coerce')
df['YR_REMODEL'] = pd.to_numeric(df['YR_REMODEL'], errors='coerce')
df['YR_BUILT'] = df['YR_BUILT'].apply(lambda x: x if x <= 2024 else np.nan)
df['YR_REMODEL'] = df['YR_REMODEL'].apply(lambda x: x if x <= 2024 else np.nan)

# Group by ZIP_CODE and calculate the mean for YR_BUILT and YR_REMODEL
condition_analysis = df.groupby('ZIP_CODE').agg(
    yr_built_mean=('YR_BUILT', 'mean'),
    yr_remodel_mean=('YR_REMODEL', 'mean')
).reset_index()

# Define thresholds for building classification
def classify_building(yr_built, yr_remodel, old_threshold=1970, new_threshold=2000):
    """
    Classify building as 'Old', 'Average', or 'New' based on YR_REMODEL or YR_BUILT.
    If YR_REMODEL exists, use it; otherwise, use YR_BUILT.
    """
    if not pd.isna(yr_remodel):
        year = yr_remodel  # Use YR_REMODEL if available
    else:
        year = yr_built  # Otherwise, use YR_BUILT
    
    if pd.isna(year):
        return 'Unknown'
    elif year <= old_threshold:
        return 'Old'
    elif year >= new_threshold:
        return 'New'
    else:
        return 'Average'
    
# Apply classification based on YR_BUILT and YR_REMODEL
condition_analysis['building_classification'] = condition_analysis.apply(
    lambda row: classify_building(row['yr_built_mean'], row['yr_remodel_mean']), axis=1)

# Print the classification results based on the YR_BUILT and YR_REMODEL means
print("Building Classification Based on YR_BUILT and YR_REMODEL:")
print(condition_analysis[['ZIP_CODE', 'yr_built_mean', 'yr_remodel_mean', 'building_classification']])

Building Classification Based on YR_BUILT and YR_REMODEL:
    ZIP_CODE  yr_built_mean  yr_remodel_mean building_classification
0     2026.0    1956.250000      2011.000000                     New
1     2108.0    1907.597015      1999.891989                 Average
2     2109.0    1924.923077      1997.760908                 Average
3     2110.0    1976.132207      2003.533708                     New
4     2111.0    1964.012129      1999.984496                 Average
5     2113.0    1914.591981      1997.117143                 Average
6     2114.0    1931.860516      1997.320281                 Average
7     2115.0    1927.726628      1996.609807                 Average
8     2116.0    1922.444645      1999.766815                 Average
9     2118.0    1935.584298      2003.057359                     New
10    2119.0    1930.757342      2003.193167                     New
11    2120.0    1937.205426      2003.518856                     New
12    2121.0    1921.378514      2003.695578 

Step 8: Include Address Details and Finalize Output
This final step merges street address details, adds overall condition labels, assigns a constant YEAR value, and saves the output to an Excel file for further analysis.

In [8]:
# Assuming 'ST_NUM' and 'ST_NAME' are part of the original dataset
# Extract those columns from the original dataset
st_num_name = df[['ZIP_CODE', 'ST_NUM', 'ST_NAME']].drop_duplicates()

# Merge 'st_num_name' with 'condition_analysis' to include 'ST_NUM' and 'ST_NAME' with the results
condition_analysis = pd.merge(condition_analysis, st_num_name, on='ZIP_CODE', how='left')

# Add overall condition label based on previous analysis
# Ensure that the column 'overall_cond_label' from earlier condition analysis is merged correctly
overall_cond_summary = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Reapply the condition label mapping function to map numeric values to condition labels
def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    else:
        return 'Unsound'

overall_cond_summary['overall_cond_label'] = overall_cond_summary['overall_cond_mean'].apply(mean_to_condition)

# Merge overall condition labels with the condition_analysis DataFrame
condition_analysis = pd.merge(condition_analysis, overall_cond_summary[['ZIP_CODE', 'overall_cond_label']], on='ZIP_CODE', how='left')

# Assign a constant value for 'YEAR'
condition_analysis['YEAR'] = 2024

# Rearranging the columns as requested
final_output = condition_analysis[['YEAR', 'ST_NUM', 'ST_NAME', 'ZIP_CODE', 'overall_cond_label', 'building_classification']]

# Save the final result to a new Excel file
output_file_path = 'Property_Assessment_2024_Output.xlsx'
final_output.to_excel(output_file_path, index=False)

print(f"File saved successfully to {output_file_path}")

File saved successfully to Property_Assessment_2024_Output.xlsx
